# Feature Engineering
**CareGuard — Hospital Readmission Risk Prediction**

This notebook prepares the raw diabetes dataset for model training by cleaning, encoding, and engineering features.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Step 1 — Load Data
Read the raw CSV, treating `?` as missing values. Create a binary target variable: `1` if the patient was readmitted within 30 days, `0` otherwise.

In [ ]:
df = pd.read_csv('../data/raw/dataset_diabetes/diabetic_data.csv', na_values='?')

df['readmitted_30'] = (df['readmitted'] == '<30').astype(int)

print(f'Loaded dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Target distribution:\n{df["readmitted_30"].value_counts()}')

## Step 2 — Drop High-Missing / Non-Informative Columns
Remove columns with excessive missing values or identifiers that carry no predictive signal.

In [ ]:
cols_to_drop = ['weight', 'payer_code', 'encounter_id', 'patient_nbr', 'medical_specialty']
df.drop(columns=cols_to_drop, inplace=True)

print(f'Shape after dropping columns: {df.shape}')

## Step 3 — Handle Missing Values
Drop rows where `race` or `gender` is missing, as these are small proportions and imputation would be misleading.

In [ ]:
before = len(df)
df.dropna(subset=['race', 'gender'], inplace=True)
after = len(df)

print(f'Dropped {before - after:,} rows with missing race/gender')
print(f'Shape after dropping: {df.shape}')

## Step 4 — Encode Age
Map age bracket strings to ordinal integers representing decade bands (0–9).

In [ ]:
age_map = {
    '[0-10)':  0,
    '[10-20)': 1,
    '[20-30)': 2,
    '[30-40)': 3,
    '[40-50)': 4,
    '[50-60)': 5,
    '[60-70)': 6,
    '[70-80)': 7,
    '[80-90)': 8,
    '[90-100)': 9
}

df['age'] = df['age'].map(age_map)

print('Age encoding complete')
print(df['age'].value_counts().sort_index())

## Step 5 — Encode Gender
Binary encode gender: `Male=1`, `Female=0`. Rows with unexpected values are dropped.

In [ ]:
df = df[df['gender'].isin(['Male', 'Female'])]
df['gender'] = (df['gender'] == 'Male').astype(int)

print('Gender encoding complete')
print(df['gender'].value_counts())

## Step 6 — One-Hot Encode Race
Expand the `race` column into binary indicator columns, dropping the first to avoid multicollinearity.

In [ ]:
race_dummies = pd.get_dummies(df['race'], prefix='race', drop_first=True)
df = pd.concat([df.drop(columns='race'), race_dummies], axis=1)

print('Race one-hot encoding complete')
print('New race columns:', list(race_dummies.columns))

## Step 7 — Engineer `prior_visits` Feature
Aggregate prior healthcare utilisation into a single feature: sum of inpatient, outpatient, and emergency visits.

In [ ]:
df['prior_visits'] = (
    df['number_inpatient'] +
    df['number_outpatient'] +
    df['number_emergency']
)

print('prior_visits feature created')
print(df['prior_visits'].describe())

## Step 8 — Encode Medication Change Columns
For each medication column, encode `'Ch'` (dosage changed) as `1` and all other values as `0`.

In [ ]:
med_cols = [
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide', 'pioglitazone',
    'rosiglitazone', 'acarbose', 'insulin', 'tolazamide'
]

for col in med_cols:
    if col in df.columns:
        df[col] = (df[col] == 'Ch').astype(int)

print('Medication change columns encoded')
print(df[med_cols].sum().sort_values(ascending=False))

## Step 9 — Encode `change` and `diabetesMed`
- `change`: `'Ch'` → `1`, else `0`
- `diabetesMed`: `'Yes'` → `1`, else `0`

In [ ]:
df['change'] = (df['change'] == 'Ch').astype(int)
df['diabetesMed'] = (df['diabetesMed'] == 'Yes').astype(int)

print('change and diabetesMed encoded')
print(df[['change', 'diabetesMed']].value_counts())

## Step 10 — Drop Original `readmitted` and Remaining String Columns
Remove the original target column and any remaining object-type columns that were not encoded.

In [ ]:
df.drop(columns=['readmitted'], inplace=True)

remaining_str_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Remaining string columns to drop: {remaining_str_cols}')
df.drop(columns=remaining_str_cols, inplace=True)

print(f'Shape after dropping string columns: {df.shape}')

## Step 11 — Save Processed Data
Write the engineered feature set to `data/processed/features.csv`.

In [ ]:
output_path = '../data/processed/features.csv'
df.to_csv(output_path, index=False)

print(f'Saved processed data to {output_path}')

## Step 12 — Final Summary

In [ ]:
print(f'Final dataset shape: {df.shape}')
print(f'\nTarget balance:')
print(df['readmitted_30'].value_counts(normalize=True).rename({0: 'Not readmitted <30d', 1: 'Readmitted <30d'}))
print(f'\nFirst few rows:')
df.head()